# Stage 1 - Phase 8: compare models and late-fuseCompares every trained Stage 1 model and searches the late fusion, using **onlythe saved `val_predictions.csv` files**. Nothing is retrained here, so thisnotebook is cheap to re-run after each new experiment.Why fusion is late rather than joint: the video branch and the forensic branchsee different inputs (clips vs native-resolution patches) and are trainedindependently, then combined at probability level:```p_final = alpha * p_video + (1 - alpha) * p_forensic```Both `alpha` and the decision threshold are searched on validation Macro-F1.Prediction correlation is reported next to the scores on purpose. On DLC-2021 ahigh score may come from a document/display shortcut, so a slightly weaker butdecorrelated model is often the better ensemble partner.

## 1. Setup

In [ ]:
from __future__ import annotationsimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchimport yaml# The package is expected to be installed with `pip install -e .` from the# repository root. The fallback keeps a fresh clone usable without installing.try:    import blackbox_detection  # noqa: F401except ModuleNotFoundError:    _root = Path.cwd()    while _root != _root.parent and not (_root / "pyproject.toml").is_file():        _root = _root.parent    sys.path.insert(0, str(_root / "src"))from blackbox_detection.utils import seed_everything, setup_loggerprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import jsonfrom blackbox_detection.stage1 import (    compare_models,    fuse_probabilities,    fused_predictions,    load_prediction_tables,    prediction_correlation,    search_all_combinations,    search_late_fusion,)from blackbox_detection.stage1.evaluator import evaluate_predictions, load_predictionslogger = setup_logger("stage1.compare_and_fuse")

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():    REPO_ROOT = REPO_ROOT.parentCONFIG_DIR = REPO_ROOT / "configs" / "stage1"OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)print("repo   :", REPO_ROOT)print("configs:", CONFIG_DIR)print("outputs:", OUTPUT_ROOT)

## 3. ConfigOnly the models that already produced a `val_predictions.csv` are compared, sothis runs at any point in the experiment sequence.

In [ ]:
# Implementation keys and the phase they belong to.CANDIDATES = {    "videomaev2_b": "V1 VideoMAEv2-B (video)",    "vjepa2_1_b": "V2 V-JEPA 2.1-B (video)",    "bayar_resnet18": "F1 Bayar + ResNet18",    "cdc": "F2 CDC binary CNN",    "chromaticity": "F3 CMA-inspired chromaticity",    "frequency": "F4 FMAG/M2FM-inspired frequency",    "lcdf": "F5 LC&DF-inspired dual stream",}VIDEO_MODELS = ("videomaev2_b", "vjepa2_1_b")PREDICTION_PATHS = {}for key in CANDIDATES:    path = OUTPUT_ROOT / key / "val_predictions.csv"    if path.is_file():        PREDICTION_PATHS[key] = path    else:        print(f"skipping {key}: no val_predictions.csv yet")if not PREDICTION_PATHS:    raise FileNotFoundError(        "No validation predictions found. Run notebook 01 and/or 02 first."    )print()print("comparing:", list(PREDICTION_PATHS))

## 4. Data`load_prediction_tables` refuses to merge files that do not cover the samevalidation videos, which is the guard against accidentally comparing modelstrained on different splits.

In [ ]:
wide = load_prediction_tables(PREDICTION_PATHS)print("validation videos:", len(wide))print("class balance:", wide["label"].value_counts().to_dict())wide.head()

## 5. ModelNo model is loaded: this notebook works purely from saved probabilities.

## 6. TrainingNot applicable in Phase 8.

## 7. Validation### 7.1 Individual models`macro_f1` is scored at each model's own optimal threshold; `macro_f1_at_0.5`shows how much of the score depends on threshold tuning.

In [ ]:
comparison = compare_models(wide)comparison.insert(1, "description", comparison["model"].map(CANDIDATES))display(comparison)

In [ ]:
# Per-run detail recorded during training, when available.rows = []for key in PREDICTION_PATHS:    summary_path = OUTPUT_ROOT / key / "summary.json"    if not summary_path.is_file():        continue    payload = json.loads(summary_path.read_text(encoding="utf-8"))    rows.append(        {            "model": key,            "best_epoch": payload.get("best_epoch"),            "val_macro_f1": payload.get("val_macro_f1"),            "best_threshold": payload.get("best_threshold"),            "subset_scores": payload.get("subset_scores"),        }    )if rows:    display(pd.DataFrame(rows))

### 7.2 Overfitting behaviourRead alongside the scores: a model peaking in epoch 1 and then degrading ismemorising DLC-2021 rather than learning a recapture representation.

In [ ]:
for key in PREDICTION_PATHS:    history_path = OUTPUT_ROOT / key / "history.csv"    if not history_path.is_file():        continue    history = pd.read_csv(history_path)    columns = [c for c in ("epoch", "train_loss", "val_macro_f1", "val_threshold") if c in history]    print(f"--- {key}")    print(history[columns].to_string(index=False))    print()

### 7.3 Prediction correlation (ensemble diversity)Two models with similar scores and low correlation are the interesting pair:they disagree on different videos, which is what late fusion can exploit.

In [ ]:
if len(PREDICTION_PATHS) >= 2:    pearson = prediction_correlation(wide, method="pearson")    spearman = prediction_correlation(wide, method="spearman")    print("Pearson correlation of prob_rerecorded:")    display(pearson.round(3))    print("Spearman correlation:")    display(spearman.round(3))    off_diagonal = pearson.where(~np.eye(len(pearson), dtype=bool))    least = off_diagonal.stack().idxmin()    print(f"least correlated pair: {least} ({off_diagonal.stack().min():.3f})")else:    print("Need at least two models for a correlation matrix.")

## 8. Results### 8.1 Late fusion searchThe weight grid is searched jointly with the threshold. For two models this isexactly the `alpha` search.

In [ ]:
available = list(PREDICTION_PATHS)video_available = [key for key in available if key in VIDEO_MODELS]forensic_available = [key for key in available if key not in VIDEO_MODELS]best_forensic = Noneif forensic_available:    forensic_scores = comparison[comparison["model"].isin(forensic_available)]    best_forensic = str(forensic_scores.iloc[0]["model"])    print("best forensic model:", best_forensic)combinations = []for video_model in video_available:    if best_forensic:        combinations.append([video_model, best_forensic])if len(video_available) >= 2:    combinations.append(list(video_available))if len(video_available) >= 2 and best_forensic:    combinations.append([*video_available, best_forensic])# Also try every pair, so a strong forensic-only ensemble is not missed.combinations += [list(pair) for pair in __import__("itertools").combinations(available, 2)]# De-duplicate while keeping order.seen = set()unique_combinations = []for group in combinations:    key = tuple(sorted(group))    if key not in seen and len(group) >= 2:        seen.add(key)        unique_combinations.append(group)print("ensembles to search:", unique_combinations)

In [ ]:
if unique_combinations:    ensembles = search_all_combinations(wide, combinations=unique_combinations, weight_step=0.05)    display(ensembles)else:    ensembles = pd.DataFrame()    print("Need at least two models to fuse.")

### 8.2 Best ensemble in detail

In [ ]:
if len(unique_combinations):    best_row = ensembles.iloc[0]    best_models = [name.strip() for name in str(best_row["ensemble"]).split("+")]    best = search_late_fusion(wide, best_models, weight_step=0.05)    print("models    :", best.models)    print("weights   :", tuple(round(weight, 3) for weight in best.weights))    print(f"threshold : {best.threshold:.4f}")    print(f"Macro-F1  : {best.macro_f1:.4f}")    print("class F1  :", best.per_class_f1)    print("singles   :", {name: round(score, 4) for name, score in best.baseline_macro_f1.items()})    print(f"gain over best single: {best.gain_over_best_single:+.4f}")    fused = fused_predictions(wide, best)    display(fused.head(10))

### 8.3 Reading the comparison* A gain near zero means the models are making the same mistakes; look at the  correlation matrix before adding compute.* A large gain from a low-scoring partner is a good sign: it is contributing  independent evidence.* Every number here is DLC-2021 validation. It does **not** establish  dashcam-domain generalisation. Keep the runner-up models and their  checkpoints: once the paired CCD re-recordings exist, VAL-CCD is what decides,  and this notebook extends to it by adding its prediction files to  `PREDICTION_PATHS`.

## 9. Save

In [ ]:
COMPARE_DIR = OUTPUT_ROOT / "comparison"COMPARE_DIR.mkdir(parents=True, exist_ok=True)comparison.to_csv(COMPARE_DIR / "model_comparison.csv", index=False)wide.to_csv(COMPARE_DIR / "val_predictions_wide.csv", index=False)if len(PREDICTION_PATHS) >= 2:    prediction_correlation(wide).to_csv(COMPARE_DIR / "prediction_correlation.csv")if len(unique_combinations):    ensembles.to_csv(COMPARE_DIR / "ensemble_search.csv", index=False)    fused.to_csv(COMPARE_DIR / "best_ensemble_predictions.csv", index=False)    (COMPARE_DIR / "best_ensemble.json").write_text(        json.dumps(            {                "models": list(best.models),                "weights": list(best.weights),                "threshold": best.threshold,                "macro_f1": best.macro_f1,                "per_class_f1": best.per_class_f1,                "single_model_macro_f1": best.baseline_macro_f1,            },            indent=2,        ),        encoding="utf-8",    )print("artefacts in", COMPARE_DIR)for path in sorted(COMPARE_DIR.iterdir()):    print("  ", path.name)